## DistilBERT vs Claude Sonnet 4.6 — Model Comparison

Side-by-side comparison of results from the DistilBERT classifier (run in Phase 1) and Claude Sonnet 4.6 via AWS Bedrock (run in Phase 4). DistilBERT returns a binary positive/negative label with a confidence score. Claude returns richer structured output including mixed sentiment, key themes, a summary, and a recommendation likelihood.


In [3]:
import boto3
import pandas as pd
from io import BytesIO

s3 = boto3.client("s3")
BUCKET = "candrews-sentiment-pipeline"

# Load both result sets
def read_s3_parquet(key):
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))

distilbert_results = read_s3_parquet("results/imdb_results.parquet")
bedrock_results = read_s3_parquet("results/bedrock_results.parquet")

print("=== DistilBERT output ===")
print(distilbert_results[["prediction_label", "confidence"]].head())

print("\n=== Claude via Bedrock output ===")
print(bedrock_results[["sentiment", "confidence", "key_themes", "summary", "would_recommend"]].head())

=== DistilBERT output ===
  prediction_label  confidence
0         POSITIVE    0.998876
1         POSITIVE    0.996983
2         NEGATIVE    0.997244
3         NEGATIVE    0.649213
4         NEGATIVE    0.998503

=== Claude via Bedrock output ===
  sentiment confidence                                       key_themes  \
0  positive       high  [acting, emotional impact, personal connection]   
1  positive       high         [comedy, spy genre, cultural references]   
2  negative       high               [stereotyping, characters, ending]   
3  positive       high                [humor, characters, storytelling]   
4  negative       high                [script, genre identity, casting]   

                                             summary  would_recommend  
0  The reviewer was deeply moved by the film's su...             True  
1  The reviewer recommends this lighthearted Fren...             True  
2  The reviewer found the movie deeply frustratin...            False  
3  The reviewe

In [4]:
# DistilBERT accuracy
label_map = {"NEGATIVE": 0, "POSITIVE": 1}
distilbert_results["predicted_int"] = distilbert_results["prediction_label"].map(label_map)
distilbert_accuracy = (distilbert_results["predicted_int"] == distilbert_results["label"]).mean()

# Claude accuracy
label_map_bedrock = {0: "negative", 1: "positive"}
bedrock_results["correct"] = bedrock_results["sentiment"] == bedrock_results["original_label"].map(label_map_bedrock)
bedrock_accuracy = bedrock_results["correct"].mean()

print(f"DistilBERT accuracy: {distilbert_accuracy:.1%} (n={len(distilbert_results)})")
print(f"Claude accuracy:     {bedrock_accuracy:.1%} (n={len(bedrock_results)})")


DistilBERT accuracy: 89.0% (n=500)
Claude accuracy:     90.0% (n=20)
